# Notebook 04 — Flashcard mode

Walks through your curriculum chunk by chunk as flashcards.

- **Front:** the topic/concept name (from the chunk's breadcrumb metadata)
- **Back:** the chunk text, optionally summarised by the model

After each card you mark it as known or unknown. Progress is saved to
`flashcard_progress.json` so it persists across sessions.

**Key concept:** the progress tracker reads the `h1/h2/h3` headers
from your `chunks.json` (populated by `chunker.py`). No extra setup needed.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

MODEL_CHOICE = "cpu_tiny"      # used to summarise the back of the card (optional)
# MODEL_CHOICE = "small_hf"
# MODEL_CHOICE = "larger_hf"
# MODEL_CHOICE = "own_model"
# MODEL_CHOICE = "none"       # skip model — just show raw chunk text on the back

CHECKPOINT_PATH   = "../teaching_assistant/checkpoints/step_005000.pt"
RAG_INDEX_DIR     = "../teaching_assistant/rag_index"
CHUNKS_JSON       = "../teaching_assistant/rag_index/chunks.json"
PROGRESS_FILE     = "flashcard_progress.json"

# Filter to a specific chapter (set to None to go through everything)
CHAPTER_FILTER = None   # e.g. "Neural Networks" or None

# How many cards to do in this session
SESSION_SIZE = 5

# Whether to use the model to summarise the back of the card
# Set False if MODEL_CHOICE == "none" or if the model is slow
SUMMARISE_BACK = False

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL LOADER — compatible with transformers >= 4.40
#
# text2text-generation was removed from the pipeline registry
# in newer transformers. We use the model + tokenizer directly
# instead — it's the same thing under the hood, just explicit.
# ═══════════════════════════════════════════════════════════
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if MODEL_CHOICE == "cpu_tiny":
    # ── flan-t5-small ──────────────────────────────────────
    # T5 is an encoder-decoder (seq2seq) model.
    # We encode the prompt with the encoder, then let the
    # decoder generate the output autoregressively.
    # ~80 MB, runs in seconds on CPU.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
    _mdl.eval()
    # Note: T5 stays on CPU (device=-1 was the old pipeline arg)

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "small_hf":
    # ── flan-t5-base ───────────────────────────────────────
    # Same architecture, 3× more parameters. Better at following
    # structured instructions (e.g. "output only JSON").
    # ~250 MB. CPU works but is slow; GPU recommended.
    from transformers import T5ForConditionalGeneration, T5Tokenizer
    _tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
    _mdl = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
    _mdl = _mdl.to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        inputs = _tok(prompt, return_tensors="pt",
                      truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
            )
        return _tok.decode(out_ids[0], skip_special_tokens=True)

elif MODEL_CHOICE == "larger_hf":
    # ── TinyLlama-1.1B-Chat ────────────────────────────────
    # Decoder-only (same family as GPT). Uses a chat template
    # so the prompt is wrapped in <|user|> / <|assistant|> tags.
    # ~2.2 GB VRAM — fits easily on a 3080 laptop.
    # Swap model_id for "microsoft/phi-2" (2.7B) for better quality.
    from transformers import AutoTokenizer, AutoModelForCausalLM
    model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    _tok = AutoTokenizer.from_pretrained(model_id)
    _mdl = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    ).to(device)
    _mdl.eval()

    def generate(prompt, max_new_tokens=300):
        # Apply the chat template so TinyLlama knows this is a user message
        messages  = [{"role": "user", "content": prompt}]
        formatted = _tok.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = _tok(formatted, return_tensors="pt").to(device)
        with torch.no_grad():
            out_ids = _mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
            )
        # Decode only the newly generated tokens (skip the prompt)
        new_ids = out_ids[0, inputs["input_ids"].shape[1]:]
        return _tok.decode(new_ids, skip_special_tokens=True)

elif MODEL_CHOICE == "own_model":
    # ── Your trained model from train.py ───────────────────
    # Uses tiktoken gpt2 BPE tokenizer (same as training).
    import sys
    sys.path.insert(0, "../teaching_assistant")
    from rag_pipeline import load_model
    import tiktoken
    _own_model, _own_tok, _own_cfg = load_model(CHECKPOINT_PATH, device)
    _enc = tiktoken.get_encoding("gpt2")

    def generate(prompt, max_new_tokens=200):
        ids = _enc.encode(prompt)
        x   = torch.tensor([ids], dtype=torch.long).to(device)
        out = _own_model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=50,
            stop_token=_enc.eot_token,
        )
        new_ids = out[0, len(ids):].tolist()
        return _enc.decode(new_ids).replace("<|endoftext|>", "").strip()

print(f"Model ready: {MODEL_CHOICE}")


In [ ]:
# ═══════════════════════════════════════════════════════════
# PROGRESS TRACKER
# Reads your chunks.json and builds a curriculum map.
# Saves progress to PROGRESS_FILE between sessions.
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "../teaching_assistant")
from progress_tracker import ProgressTracker

tracker = ProgressTracker(CHUNKS_JSON, PROGRESS_FILE)

# Show what chapters are available
print("Available chapters:", tracker.chapters())
tracker.print_progress()

In [ ]:
# ═══════════════════════════════════════════════════════════
# FLASHCARD SESSION
# Each iteration:
#   1. Picks next chunk by priority (unseen first, then lowest accuracy)
#   2. Shows the FRONT (topic name)
#   3. Waits for you to press Enter to reveal the BACK (content)
#   4. Asks: did you know it? (y/n)
#   5. Records and repeats
# ═══════════════════════════════════════════════════════════
import json

# Load chunk texts (we need them for the back of the card)
with open(CHUNKS_JSON) as f:
    all_chunks = json.load(f)


def make_back(chunk_text, topic):
    """Create the back of the flashcard from the chunk text.
    Optionally summarises with the model for a cleaner explanation."""
    if SUMMARISE_BACK and generate is not None:
        prompt = f"Summarise this in 2-3 sentences for a student:\n{chunk_text[:500]}"
        return generate(prompt)
    else:
        # Just show the raw text, truncated for readability
        return chunk_text[:600] + ("..." if len(chunk_text) > 600 else "")


def run_flashcards(n=SESSION_SIZE, chapter=CHAPTER_FILTER):
    known = 0

    for card_num in range(1, n + 1):
        result = tracker.next_chunk(chapter=chapter)
        if result is None:
            print("No more chunks available for this filter.")
            break

        chunk_id, record = result
        chunk_text = all_chunks[chunk_id]["text"]

        print(f"\n{'═'*55}")
        print(f"Card {card_num}/{n}   |   {record.breadcrumb}")
        print(f"(Seen {record.times_seen}× | Accuracy {record.accuracy:.0%})")
        print('─'*55)

        # FRONT
        print(f"\n  FRONT:  {record.topic}\n")
        input("  [ Press Enter to reveal the back ] ")

        # BACK
        back = make_back(chunk_text, record.topic)
        print(f"\n  BACK:\n")
        print("  " + back.replace("\n", "\n  "))

        # Self-assessment
        while True:
            ans = input("\n  Did you know it? (y/n): ").strip().lower()
            if ans in ("y", "n"):
                break
            print("  Type y or n")

        correct = ans == "y"
        tracker.record(chunk_id, correct)
        if correct:
            known += 1
            print("  ✓ Marked as known")
        else:
            print("  ✗ Will review again soon")

    print(f"\n{'═'*55}")
    print(f"Session complete: {known}/{card_num} known")
    tracker.print_progress()


run_flashcards()

In [ ]:
# ═══════════════════════════════════════════════════════════
# INSPECT WEAK SPOTS
# Shows which topics you struggle with most.
# ═══════════════════════════════════════════════════════════
weak = tracker.weak_topics(n=5)
print("Your 5 weakest topics:")
for topic, acc in weak:
    print(f"  {topic:<40} {acc:.0%} accuracy")